In [7]:
import torch
from torch_geometric.nn import GraphSAGE
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm

In [8]:
# ============================================
# 1. Подготовка данных
# ============================================
df = pd.read_csv('data/edges/IW_edges.csv')

all_ids = pd.concat([df['id_entity_1'], df['id_entity_2']]).unique()
id_to_idx = {id_: idx for idx, id_ in enumerate(all_ids)}

src_idx = [id_to_idx[id_] for id_ in df['id_entity_1']]
dst_idx = [id_to_idx[id_] for id_ in df['id_entity_2']]

edge_index = torch.tensor([src_idx, dst_idx], dtype=torch.long)

num_nodes = len(all_ids)
x = torch.randn(num_nodes, 16)

data = Data(x=x, edge_index=edge_index)

In [9]:
# ============================================
# 2. Настройка устройства (GPU/CPU)
# ============================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Память GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Переносим данные на GPU
data = data.to(device)

Используемое устройство: cuda
GPU: AMD Radeon RX 6800 XT
Память GPU: 17.16 GB


In [10]:
# ============================================
# 3. Загрузчик
# ============================================
loader = LinkNeighborLoader(
    data, 
    num_neighbors=[10, 10],
    batch_size=1024,
    neg_sampling_ratio=1.0,
    shuffle=True
)

# ============================================
# 4. Модель (переносим на GPU)
# ============================================
model = GraphSAGE(in_channels=16, hidden_channels=32, num_layers=2, out_channels=32)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

/home/askakolbaska/graph_LP/.venv/lib/python3.12/site-packages/torch_geometric/loader/link_neighbor_loader.py:252: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


In [11]:
# ============================================
# 5. Обучение
# ============================================
def train():
    model.train()
    total_loss = 0
    total_examples = 0
    
    for batch in loader:
        # batch автоматически на правильном устройстве (т.к. data на GPU)
        optimizer.zero_grad()
        
        h = model(batch.x, batch.edge_index)
        
        src, dst = batch.edge_label_index
        z_src = h[src]
        z_dst = h[dst]
        
        pred = (z_src * z_dst).sum(dim=-1)
        
        # edge_label тоже автоматически на GPU
        loss = F.binary_cross_entropy_with_logits(pred, batch.edge_label.float())
        
        loss.backward()
        optimizer.step()
        
        total_loss += float(loss) * batch.edge_label.size(0)
        total_examples += batch.edge_label.size(0)
    
    return total_loss / total_examples

# Цикл обучения
for epoch in range(10):
    loss = train()
    print(f"Эпоха {epoch:02d}, Потеря: {loss:.4f}")

ImportError: 'NeighborSampler' requires either 'pyg-lib' or 'torch-sparse'

In [ ]:

# 2. Загрузчик для семплирования соседей (критично для GraphSAGE)
loader = LinkNeighborLoader(
    data, 
    num_neighbors=[10, 10], # Сколько соседей брать на каждом слое
    batch_size=1024,
    neg_sampling_ratio=1.0, # Генерация отрицательных примеров для link prediction
    shuffle=True
)

# 3. Модель
model = GraphSAGE(in_channels=16, hidden_channels=32, num_layers=2, out_channels=32)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def train():
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        
        # Получаем эмбеддинги для узлов в батче
        h = model(batch.x, batch.edge_index)
        
        # В PyG LinkNeighborLoader автоматически добавляет edge_label (1 или 0)
        # и edge_label_index (пары узлов для проверки)
        src, dst = batch.edge_label_index
        z_src = h[src]
        z_dst = h[dst]
        
        # Скоринг связи (косинусное сходство или скалярное произведение)
        pred = (z_src * z_dst).sum(dim=-1)
        
        loss = torch.nn.functional.binary_cross_entropy_with_logits(pred, batch.edge_label.float())
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * batch.num_edges
    return total_loss / len(loader.dataset)

# Простой цикл обучения
for epoch in range(5):
    loss = train()
    print(f"Эпоха {epoch}, Потеря: {loss:.4f}")

In [ ]:


def evaluate_ranking(model, data, test_edges, k=10):
    """
    data: полный объект графа (для доступа ко всем узлам при ранжировании)
    test_edges: тензор [num_test, 2] с парами узлов для теста
    """
    model.eval()
    
    # 1. Получаем эмбеддинги ВСЕХ узлов графа один раз
    # Важно: используем весь граф для получения контекста, даже если обучались на семплах
    with torch.no_grad():
        h = model(data.x, data.edge_index)
    
    src_nodes = test_edges[:, 0]
    dst_nodes = test_edges[:, 1]
    
    h_src = h[src_nodes]
    h_dst_true = h[dst_nodes]
    
    mrr_list = []
    hits_list = []
    
    # 2. Цикл по тестовым ребрам (можно ускорить батчами, но для наглядности по одному)
    for i in tqdm(range(len(test_edges)), desc="Ranking Eval"):
        z_src = h_src[i]
        z_true = h_dst_true[i]
        
        # Считаем скоры для всех возможных объектов (или большой выборки негативов)
        # Скалярное произведение источника со всеми узлами графа
        scores = torch.matmul(z_src.unsqueeze(0), h.t()).squeeze(0)
        
        # Маскируем само ребро и существующие связи, если нужно (опционально)
        # Здесь для простоты считаем, что ранг определяется только величиной скора
        
        # Сортируем убываю
        _, rank_indices = torch.sort(scores, descending=True)
        
        # Находим ранг истинного объекта (индекс + 1, так как ранги с 1)
        rank = (rank_indices == dst_nodes[i]).nonzero(as_tuple=True)[0].item() + 1
        
        # MRR = 1 / rank
        mrr_list.append(1.0 / rank)
        
        # Hits@K = 1 если ранг <= K, иначе 0
        hits_list.append(1.0 if rank <= k else 0.0)

    return sum(mrr_list) / len(mrr_list), sum(hits_list) / len(mrr_list)

# Пример использования:
# mrr, hits10 = evaluate_ranking(model, data, test_edge_index, k=10)
# print(f"MRR: {mrr:.4f}, Hits@10: {hits10:.4f}")